In [1]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao

,Ano,Base,Tabelas
0,2025,2025.duckdb,8
1,2026,2026.duckdb,11


In [3]:
# ==========================
# 1. Configuração
# ==========================
meses = {
    1: "Jan",
    2: "Fev",
    3: "Mar",
    4: "Abr",
    5: "Mai",
    6: "Jun",
    7: "Jul",
    8: "Ago",
    9: "Set",
    10: "Out",
    11: "Nov",
    12: "Dez",
}

tabelas_paletes = {}


# ==========================
# 2. Processar anos
# ==========================
for ano in ANOS:

    df_ano = CON[ano].execute(f"""
        SELECT
            CODACT,
            ACTIVIDAD AS CLIENTE,
            FENTREGA,
            TRY_CAST(
                REPLACE(PALETS, ',', '.')
                AS DOUBLE
            ) AS PALETS
        FROM inform_27_{ano}
        WHERE Gestion = ?
    """, [GESTAO]).df()

    # ==========================
    # 3. Extrair mês
    # ==========================
    df_ano["MES"] = pd.to_numeric(
        df_ano["FENTREGA"]
        .astype("string")
        .str.strip()
        .str[4:6],
        errors="coerce"
    )

    # ==========================
    # 4. Criar tabela mensal
    # ==========================
    tabela = (
        df_ano
        .pivot_table(
            index=["CODACT", "CLIENTE"],
            columns="MES",
            values="PALETS",
            aggfunc="sum",
            fill_value=0
        )
        .reindex(
            columns=range(1, 13),
            fill_value=0
        )
        .rename(columns=meses)
        .reset_index()
    )

    # ==========================
    # 5. Formatar paletes
    # ==========================
    colunas_meses = list(meses.values())

    tabela[colunas_meses] = (
        tabela[colunas_meses]
        .round(0)
        .astype(int)
    )

    tabelas_paletes[ano] = tabela


# ==========================
# 6. Mostrar 2026
# ==========================
tabelas_paletes[2026]

MES,CODACT,CLIENTE,Jan,Fev,Mar,Abr,Mai,Jun,Jul,Ago,Set,Out,Nov,Dez
0,011,DANONE PORTUGAL,2939,2473,14378,15000,16625,15909,16358,18342,3509,0,0,0
1,013,DANONE PORTUGAL CAPILAR,2982,2914,4166,3990,3609,3173,3365,3046,656,0,0,0
2,015,TROPICANA,22,8,109,92,154,256,383,234,55,0,0,0
3,028,PLN,0,0,0,0,1,9,18,25,6,0,0,0
4,068,FRIESLANDCAMPIÑA,1,0,7,0,15,0,0,0,0,0,0,0
5,074,NUTRICIA,1298,858,5599,4934,4221,3778,4169,4181,1164,0,0,0
6,084,DANONE PORTUGAL TLD,495,363,528,495,528,1695,807,528,132,0,0,0
7,116,ORANGINA SCHWEPPES ESPAÑA BASE,0,0,0,0,3,6,5,8,0,0,0,0
8,117,ORANGINA SCHWEPPES,0,8,72,88,84,192,288,170,23,0,0,0
9,118,ORANGNA SCHWEPPES PORTUGAL,59,45,482,447,422,501,594,588,121,0,0,0
